# Pill Detection and Counting with YOLOv8

This notebook follows the project workflow: download the Roboflow dataset, train YOLOv8, evaluate the trained checkpoint, run predictions, and optionally export the model. Set `ROBOFLOW_API_KEY` in your environment before running the notebook.

In [ ]:
from pathlib import Path
import os
import torch
import ultralytics
from roboflow import Roboflow
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd().resolve()
DATASETS_DIR = PROJECT_ROOT / "datasets"
RUNS_DIR = PROJECT_ROOT / "runs"
DATASETS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")

## 1. Download the dataset from Roboflow

The API key is read from the `ROBOFLOW_API_KEY` environment variable. Never paste a private key into a notebook or commit it to GitHub.

In [ ]:
api_key = os.environ.get("ROBOFLOW_API_KEY")
if not api_key:
    raise RuntimeError("Set the ROBOFLOW_API_KEY environment variable before running this cell.")

rf = Roboflow(api_key=api_key)
project = rf.workspace("seblful").project("pills-detection-s9ywn")
dataset = project.version(19).download("yolov8", location=str(DATASETS_DIR))

data_yaml = Path(dataset.location) / "data.yaml"
if not data_yaml.is_file():
    raise FileNotFoundError(f"Roboflow data.yaml not found: {data_yaml}")

print("Dataset location:", dataset.location)
print("Dataset config:", data_yaml)

## 2. Train YOLOv8

This uses the pretrained YOLOv8 small checkpoint and the same core training settings shown in the supplied notebook: 100 epochs, 640-pixel images, and batch size 8. Adjust the batch size if your GPU has limited memory.

In [ ]:
ultralytics.checks()

model = YOLO("yolov8s.pt")
model.train(
    data=str(data_yaml),
    epochs=100,
    imgsz=640,
    batch=8,
    project=str(RUNS_DIR),
    name="pill_detection",
    exist_ok=True,
    device=0 if torch.cuda.is_available() else "cpu",
    workers=0,
)

best_model_path = RUNS_DIR / "pill_detection" / "weights" / "best.pt"
if not best_model_path.is_file():
    raise FileNotFoundError(f"Expected best checkpoint was not created: {best_model_path}")

print("Best checkpoint:", best_model_path)

## 3. Validate the best checkpoint

The validation metrics are produced by the model and dataset actually used in this run. Record the real values from your run rather than entering estimated performance in the README.

In [ ]:
trained_model = YOLO(str(best_model_path))
metrics = trained_model.val(data=str(data_yaml))
print(metrics)

## 4. Run predictions on the test images

The prediction output is saved by Ultralytics under the project runs directory. The number of detected boxes in each image can be used as an image-level count; validate this behavior against your own annotated test images.

In [ ]:
test_images = Path(dataset.location) / "test" / "images"
if test_images.is_dir():
    predictions = trained_model.predict(
        source=str(test_images),
        conf=0.40,
        save=True,
        project=str(RUNS_DIR),
        name="pill_predictions",
        exist_ok=True,
    )
    print(f"Processed {len(predictions)} test image(s).")
else:
    print(f"No test/images directory found at {test_images}; skipping test prediction.")

## 5. Optional: export to ONNX

Run this cell only when you need an ONNX artifact for a compatible deployment environment.

In [ ]:
onnx_path = trained_model.export(format="onnx")
print("Exported model:", onnx_path)

## Notes

- Downloaded datasets, model checkpoints, and generated runs are intentionally excluded from version control.
- Check the Roboflow dataset's license and attribution requirements before redistributing it.
- This is an educational computer-vision project, not a medical device.